<h1>Problem 1: Generating Random Boolean Functions</h1>

The Deutsch-Jozsa algorithm is designed to work with Boolean functions - functions that take Boolean inputs (True/False) and return a Boolean output. For this problem, we're dealing with functions that take 4 Boolean arguments as input.

These functions are guaranteed to be one of two types. A constant function always returns the same value no matter what inputs you give it - either it always returns True, or it always returns False. A balanced function returns True for exactly half of the possible input combinations and False for the other half.

Since we have 4 Boolean inputs, there are 16 possible combinations of inputs (2^4 = 16). This means a balanced function will return True for 8 of those combinations and False for the other 8.

For constant functions, there are only 2 possibilities - the always-True function and the always-False function. For balanced functions, there are many more possibilities since we need to choose which 8 inputs return True.

The goal of this problem is to write a Python function that randomly generates one of these constant or balanced functions and returns it so it can be called and tested.

In [13]:
import random

def random_constant_balanced():
    """
    Generate a random Boolean function that's either constant or balanced.
    
    For 4 input bits, we have 16 possible input combinations.
    - Constant: all 16 inputs map to same output (0 or 1)
    - Balanced: exactly 8 inputs map to 1, other 8 map to 0
    
    Returns a callable function f(a, b, c, d) that returns a boolean.
    """
    # Decide function type: 0 = constant, 1 = balanced
    function_type = random.randint(0, 1)
    
    if function_type == 0:
        # Constant function - just pick one value and always return it
        constant_output = random.choice([False, True])
        
        def generated_function(a, b, c, d):
            return constant_output
        
        return generated_function
    
    else:
        # Balanced function - need to pick which 8 inputs return True
        
        # Build all 16 possible 4-bit input combinations
        all_inputs = [(a, b, c, d) 
                      for a in [False, True]
                      for b in [False, True]
                      for c in [False, True]
                      for d in [False, True]]
        
        # Shuffle and split: first 8 return True, last 8 return False
        random.shuffle(all_inputs)
        inputs_returning_true = all_inputs[:8]
        
        # Store as set for O(1) lookup
        true_set = set(inputs_returning_true)
        
        def generated_function(a, b, c, d):
            return (a, b, c, d) in true_set
        
        return generated_function

### Test 1: Boolean Output Validation

First, we need to verify the most basic requirement - that our function actually returns boolean values. This test checks all 16 possible input combinations to make sure every single output is either True or False.

In [14]:
# Test 1: Verify function returns valid boolean outputs
test_func = random_constant_balanced()

valid_outputs = True
for a in [False, True]:
    for b in [False, True]:
        for c in [False, True]:
            for d in [False, True]:
                result = test_func(a, b, c, d)
                if not isinstance(result, bool):
                    valid_outputs = False
                    break

assert valid_outputs, "Function must return boolean values only"
print("Test 1 passed: All outputs are boolean")

Test 1 passed: All outputs are boolean


### Test 2: Constant or Balanced Verification

The Deutsch-Jozsa algorithm only works if the function is guaranteed to be either constant or balanced - there's no in-between. This test generates many random functions and counts how many times True appears in their outputs. A valid function will have exactly 0, 8, or 16 True outputs (constant-False, balanced, or constant-True). If we ever get something like 3 or 12 True outputs, our generator is broken.

In [15]:
# Test 2: Verify generated functions are either constant OR balanced (never anything else)
def check_function_validity(func):
    """Check if function is constant or balanced by counting True outputs."""
    true_outputs = 0
    total_inputs = 0
    
    for a in [False, True]:
        for b in [False, True]:
            for c in [False, True]:
                for d in [False, True]:
                    if func(a, b, c, d):
                        true_outputs += 1
                    total_inputs += 1
    
    # Valid if: all False (0), all True (16), or exactly half (8)
    return true_outputs in [0, 8, 16]

# Test many random functions
test_iterations = 150
all_valid = True
for i in range(test_iterations):
    generated = random_constant_balanced()
    if not check_function_validity(generated):
        all_valid = False
        break

assert all_valid, f"Generated invalid function (not constant or balanced)"
print(f"Test 2 passed: All {test_iterations} functions are valid (constant or balanced)")

Test 2 passed: All 150 functions are valid (constant or balanced)


### Test 3: Random Distribution Check

Since we're randomly choosing between constant and balanced functions, we should see both types appearing with roughly equal probability over many trials. If we ran 1000 tests and got 950 constant functions and only 50 balanced ones, something would be wrong with our random selection logic. This test verifies the 50/50 split is working as expected.

In [16]:
# Test 3: Check random distribution - should get both types with reasonable probability
constant_functions = 0
balanced_functions = 0
sample_size = 800

for trial in range(sample_size):
    func = random_constant_balanced()
    
    # Count True outputs
    num_true = 0
    for a in [False, True]:
        for b in [False, True]:
            for c in [False, True]:
                for d in [False, True]:
                    if func(a, b, c, d):
                        num_true += 1
    
    # Classify the function
    if num_true == 0 or num_true == 16:
        constant_functions += 1
    elif num_true == 8:
        balanced_functions += 1

# Should be roughly 50/50 split (allow 40-60% range)
constant_percentage = (constant_functions / sample_size) * 100
balanced_percentage = (balanced_functions / sample_size) * 100

assert 40 <= constant_percentage <= 60, f"Constant functions: {constant_percentage:.1f}% (expected ~50%)"
assert 40 <= balanced_percentage <= 60, f"Balanced functions: {balanced_percentage:.1f}% (expected ~50%)"

print(f"Test 3 passed: Distribution check")
print(f"  - Constant: {constant_percentage:.1f}%")
print(f"  - Balanced: {balanced_percentage:.1f}%")

Test 3 passed: Distribution check
  - Constant: 55.6%
  - Balanced: 44.4%


### Test 4: Both Constant Types Generated

For constant functions, there are only two possibilities - a function that always returns True, or one that always returns False. This test makes sure our generator can produce both types. It keeps generating functions until it has seen at least one all-True and one all-False constant function, proving our random constant value selection is working properly.

In [17]:
# Test 4: Verify we can generate both types of constant functions (all-True and all-False)
found_all_true = False
found_all_false = False
max_attempts = 400

for attempt in range(max_attempts):
    func = random_constant_balanced()
    
    # Get first output to check if constant
    first_output = func(False, False, False, False)
    
    # Check if all outputs match first output
    is_constant = True
    for a in [False, True]:
        for b in [False, True]:
            for c in [False, True]:
                for d in [False, True]:
                    if func(a, b, c, d) != first_output:
                        is_constant = False
                        break
    
    if is_constant:
        if first_output == True:
            found_all_true = True
        else:
            found_all_false = True
    
    if found_all_true and found_all_false:
        break

assert found_all_true, f"Never generated constant-True function in {max_attempts} attempts"
assert found_all_false, f"Never generated constant-False function in {max_attempts} attempts"
print(f"Test 4 passed: Both constant types generated (all-True and all-False)")

Test 4 passed: Both constant types generated (all-True and all-False)


## Test Results Analysis

The tests above confirm several important properties of the random Boolean function generator:

**Function correctness:** All generated functions strictly follow the Deutsch-Jozsa requirements. Every function is either constant (returns the same value for all inputs) or balanced (returns True for exactly half the inputs). We never generate invalid functions that fall somewhere in between.

**Random distribution:** Over hundreds of trials, we see roughly equal numbers of constant and balanced functions being generated. This 50/50 split is important because it means our generator isn't biased toward one type. In a real quantum computing experiment, we'd want to test the algorithm against both types equally.

**Complete coverage:** The generator successfully produces both varieties of constant functions - those that always return True and those that always return False. This proves our random selection logic works correctly for all cases.

These validated functions can now be used to test the Deutsch-Jozsa quantum algorithm in the later problems. The algorithm should correctly identify whether each function is constant or balanced using only a single query to the quantum oracle, demonstrating quantum advantage over classical approaches.

## Example Function Outputs

To understand what these functions actually look like, let's generate a few examples and display their complete truth tables.

In [18]:
# Generate and display a constant function
print("=== Example 1: Constant Function ===")
const_func = random_constant_balanced()

print("\nInput (a, b, c, d) -> Output")
print("-" * 30)
for a in [False, True]:
    for b in [False, True]:
        for c in [False, True]:
            for d in [False, True]:
                output = const_func(a, b, c, d)
                print(f"({int(a)}, {int(b)}, {int(c)}, {int(d)}) -> {output}")

# Count the outputs
outputs = [const_func(a, b, c, d) 
           for a in [False, True]
           for b in [False, True]
           for c in [False, True]
           for d in [False, True]]
true_count = sum(outputs)
print(f"\nResult: {true_count} True outputs out of 16")
print(f"Function type: Constant ({'all True' if true_count == 16 else 'all False'})")

=== Example 1: Constant Function ===

Input (a, b, c, d) -> Output
------------------------------
(0, 0, 0, 0) -> True
(0, 0, 0, 1) -> True
(0, 0, 1, 0) -> True
(0, 0, 1, 1) -> True
(0, 1, 0, 0) -> True
(0, 1, 0, 1) -> True
(0, 1, 1, 0) -> True
(0, 1, 1, 1) -> True
(1, 0, 0, 0) -> True
(1, 0, 0, 1) -> True
(1, 0, 1, 0) -> True
(1, 0, 1, 1) -> True
(1, 1, 0, 0) -> True
(1, 1, 0, 1) -> True
(1, 1, 1, 0) -> True
(1, 1, 1, 1) -> True

Result: 16 True outputs out of 16
Function type: Constant (all True)


In [19]:
# Generate and display a balanced function (keep trying until we get one)
print("=== Example 2: Balanced Function ===")

# Keep generating until we get a balanced function
while True:
    balanced_func = random_constant_balanced()
    
    # Check if it's balanced
    outputs = [balanced_func(a, b, c, d) 
               for a in [False, True]
               for b in [False, True]
               for c in [False, True]
               for d in [False, True]]
    
    if sum(outputs) == 8:  # Balanced has exactly 8 True outputs
        break

print("\nInput (a, b, c, d) -> Output")
print("-" * 30)
for a in [False, True]:
    for b in [False, True]:
        for c in [False, True]:
            for d in [False, True]:
                output = balanced_func(a, b, c, d)
                print(f"({int(a)}, {int(b)}, {int(c)}, {int(d)}) -> {output}")

true_count = sum(outputs)
print(f"\nResult: {true_count} True outputs out of 16")
print("Function type: Balanced")

=== Example 2: Balanced Function ===

Input (a, b, c, d) -> Output
------------------------------
(0, 0, 0, 0) -> False
(0, 0, 0, 1) -> False
(0, 0, 1, 0) -> True
(0, 0, 1, 1) -> False
(0, 1, 0, 0) -> True
(0, 1, 0, 1) -> False
(0, 1, 1, 0) -> True
(0, 1, 1, 1) -> False
(1, 0, 0, 0) -> False
(1, 0, 0, 1) -> False
(1, 0, 1, 0) -> False
(1, 0, 1, 1) -> True
(1, 1, 0, 0) -> True
(1, 1, 0, 1) -> True
(1, 1, 1, 0) -> True
(1, 1, 1, 1) -> True

Result: 8 True outputs out of 16
Function type: Balanced


## Problem 1 Summary

I've successfully implemented a random Boolean function generator that produces functions satisfying the Deutsch-Jozsa algorithm's requirements. The generator creates either constant functions (always returning the same value) or balanced functions (returning True for exactly half of all possible inputs) with equal probability.

**Key achievements:**

The implementation correctly handles all cases - both types of constant functions (all-True and all-False) as well as the large variety of possible balanced functions. The comprehensive test suite validates that every generated function adheres to the strict constant-or-balanced requirement, with no edge cases or invalid outputs.

The random distribution testing confirms the generator produces both function types with roughly 50/50 probability, which is important for fairly testing the quantum algorithm later. The concrete examples demonstrate what these functions actually look like in practice, showing the clear distinction between constant behavior (identical outputs for all inputs) and balanced behavior (equal split of True and False outputs).

**Looking ahead:**

These validated Boolean functions will serve as test cases for the Deutsch-Jozsa quantum algorithm in the upcoming problems. The quantum algorithm's key advantage is that it can determine whether a function is constant or balanced with a single query to a quantum oracle, whereas a classical algorithm would need to check multiple inputs in the worst case. Problem 2 will explore this classical approach to establish the baseline I'm trying to improve upon.

<h1>Problem 2: Classical Testing for Function Type</h1>

## Problem 2: Classical Testing for Function Type

Now that I can generate random Boolean functions, I need to figure out how a classical (non-quantum) algorithm would determine whether a function is constant or balanced.

**The classical approach:**

For a function with 4 input bits, there are 16 possible input combinations. To figure out if a function is constant or balanced with absolute certainty, a classical algorithm has to check enough inputs to be sure.

**Worst case:**

Think about the worst case scenario. Say I'm checking inputs one by one and trying to figure out what type of function it is. I might get really unlucky - if I check the first input and get True, the function could be constant-True or balanced. If I check a second input and also get True, I still can't tell the difference. 

In fact, I could check 8 inputs and get True from all of them, and the function could still be either constant-True (where all 16 inputs return True) or balanced (where exactly 8 return True). It's only when I check the 9th input that I can know for certain - if it returns True then it must be constant, but if it returns False then it's balanced.

So a classical algorithm needs at least 9 queries (more than half of the 16 total inputs) in the worst case to guarantee it gets the right answer.

**What I'm implementing:**

I'll write a classical testing function that queries a Boolean function and figures out what type it is, while keeping track of how many queries it needed to make.

In [20]:
def determine_constant_balanced(f):
    """
    Figure out if a Boolean function is constant or balanced by querying it.
    
    Strategy: Keep querying inputs until we have enough information.
    - If we see both True and False outputs, it must be balanced
    - If we see the same output 9 times, it must be constant
      (since balanced functions have exactly 8 of each value)
    
    Args:
        f: A function taking 4 boolean arguments, returning a boolean
    
    Returns:
        "constant" or "balanced"
    """
    # Track what outputs we've seen and how many of each
    true_count = 0
    false_count = 0
    queries_made = 0
    
    # Query inputs one by one until we know the answer
    for a in [False, True]:
        for b in [False, True]:
            for c in [False, True]:
                for d in [False, True]:
                    # Query the function
                    output = f(a, b, c, d)
                    queries_made += 1
                    
                    # Update counts
                    if output:
                        true_count += 1
                    else:
                        false_count += 1
                    
                    # Check if we can determine the answer yet
                    # If we've seen both outputs, it's balanced
                    if true_count > 0 and false_count > 0:
                        return "balanced"
                    
                    # If we've seen 9 of the same output, it must be constant
                    if true_count == 9 or false_count == 9:
                        return "constant"
    
    # If we get here, all 16 were the same - constant
    return "constant"

In [21]:
# Test 1: Verify the function returns valid type strings
test_func = random_constant_balanced()
result = determine_constant_balanced(test_func)

assert result in ["constant", "balanced"], f"Expected 'constant' or 'balanced', got '{result}'"
print(" Test 1 passed: Returns valid type string")

 Test 1 passed: Returns valid type string


In [22]:
# Test 2: Verify correct classification on many random functions
correct_classifications = 0
total_tests = 200

for trial in range(total_tests):
    # Generate a random function
    func = random_constant_balanced()
    
    # Determine its actual type by checking all outputs
    all_outputs = [func(a, b, c, d)
                   for a in [False, True]
                   for b in [False, True]
                   for c in [False, True]
                   for d in [False, True]]
    
    true_count = sum(all_outputs)
    actual_type = "constant" if true_count in [0, 16] else "balanced"
    
    # Test our classical function
    determined_type = determine_constant_balanced(func)
    
    if determined_type == actual_type:
        correct_classifications += 1

assert correct_classifications == total_tests, f"Only {correct_classifications}/{total_tests} correct"
print(f" Test 2 passed: Correctly classified {correct_classifications}/{total_tests} random functions")

 Test 2 passed: Correctly classified 200/200 random functions


In [23]:
# Test 3: Verify correct classification with manually created balanced functions
def balanced_example_1(a, b, c, d):
    # Returns True for first 8 inputs when enumerated in order
    inputs_true = [
        (False, False, False, False),
        (False, False, False, True),
        (False, False, True, False),
        (False, False, True, True),
        (False, True, False, False),
        (False, True, False, True),
        (False, True, True, False),
        (False, True, True, True),
    ]
    return (a, b, c, d) in inputs_true

def balanced_example_2(a, b, c, d):
    # Returns True when odd number of inputs are True
    return (a + b + c + d) % 2 == 1

# Test both balanced functions
result1 = determine_constant_balanced(balanced_example_1)
result2 = determine_constant_balanced(balanced_example_2)

assert result1 == "balanced", f"balanced_example_1 should be balanced, got {result1}"
assert result2 == "balanced", f"balanced_example_2 should be balanced, got {result2}"
print(f" Test 3 passed: Both functions identified as '{result1}' and '{result2}'")

 Test 3 passed: Both functions identified as 'balanced' and 'balanced'


<h1>Problem 3: Quantum Oracles</h1>

<h1>Problem 4: Deutsch's Algorithm with Qiskit</h1>

<h1>Problem 5: Scaling to the Deutsch–Jozsa Algorithm</h1>